# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library, following Croissant schema best practices and referencing all data entities using their `@id`.

### Dataset Source
The dataset schema is accessible at:  
`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata using `mlcroissant`. This will also parse all record sets, fields, and column definitions for further exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}\n\nDescription: {metadata.description}")


## 2. Data Overview
Review the available record sets (`cr:RecordSet`), their fields, and list their `@id` values. These IDs will be used for all subsequent data extraction steps.

**Note:** For best practices, all referencing must use the `@id` identifiers.

In [ ]:
# List the available record sets and their field @id's
if not metadata.record_sets:
    print("No record sets (`cr:RecordSet`) found in the dataset metadata.")
else:
    print("Record Sets in the dataset:")
    for rs in metadata.record_sets:
        print(f"  RecordSet @id: {rs.id}  (name: {rs.name})")
        if hasattr(rs, 'fields') and rs.fields:
            print("    Fields:")
            for f in rs.fields:
                print(f"      Field @id: {f.id}  (name: {f.name})")
        print()
    print(f"Total number of record sets: {len(metadata.record_sets)}")

## 3. Data Extraction
Load one or more record sets into pandas DataFrames using their `@id`. Each dataframe column corresponds to a field `@id`.

Below, we demonstrate how to load all available record sets. If there are none, this cell will indicate that as well.

In [ ]:
# Prepare to extract all record sets (using their @id)
record_set_ids = [rs.id for rs in getattr(metadata, 'record_sets', [])] if hasattr(metadata, 'record_sets') else []
dataframes = {}

if not record_set_ids:
    print('No record sets are defined in the Croissant schema for this dataset.')
else:
    print(f"Extracting {len(record_set_ids)} record set(s) by @id:")
    for record_set_id in record_set_ids:
        print(f"  - {record_set_id}")
        try:
            records = list(dataset.records(record_set=record_set_id))
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"    Loaded {len(df)} records with columns: {list(df.columns)}")
        except Exception as e:
            print(f"    Could not load records for {record_set_id}: {e}")
    # Display basic info about the first dataframe, if any
    if dataframes:
        first_id = record_set_ids[0]
        print(f"\nSample records from first record set ({first_id}):")
        display(dataframes[first_id].head())

## 4. Exploratory Data Analysis (EDA)

Explore numeric fields, filtering and transforming data for analysis. All field references should use their `@id` value.

If there is at least one record set with numeric fields, we will demonstrate filtering and normalizing the first such field.

In [ ]:
# EDA: Filter and normalize numeric field (if available)
import numpy as np

if not dataframes:
    print('No dataframes loaded (no record sets).')
else:
    # Choose the first record set with non-empty records
    for rs in getattr(metadata, 'record_sets', []):
        rs_id = rs.id
        df = dataframes[rs_id]
        if df.empty:
            continue
        # Try to find a numeric field by @id (using inferred dtype in DataFrame or by metadata)
        numeric_field_id = None
        group_field_id = None
        for field in getattr(rs, 'fields', []):
            # Field id:
            f_id = field.id
            # Check if column exists and seems numeric
            if f_id in df.columns:
                series = df[f_id]
                # Try conversion to numeric to test
                try:
                    ser = pd.to_numeric(series)
                    if ser.notnull().sum() > 0:
                        numeric_field_id = f_id
                        break
                except Exception:
                    continue
        for field in getattr(rs, 'fields', []):
            # Use a non-numeric field as group if possible
            if field.id in df.columns and field.id != numeric_field_id:
                if df[field.id].dtype == object:
                    group_field_id = field.id
                    break
        if numeric_field_id:
            print(f"Using record set @id: {rs_id}")
            print(f"Analyzing numeric field @id: {numeric_field_id}")
            threshold = df[numeric_field_id].dropna().quantile(0.5) if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
            # Attempt conversion to numeric if not already
            if not pd.api.types.is_numeric_dtype(df[numeric_field_id]):
                df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
            filtered_df = df[df[numeric_field_id] > threshold]
            print(f"Filtered records with {numeric_field_id} > {threshold}:")
            display(filtered_df.head())

            # Normalization
            mean = filtered_df[numeric_field_id].mean()
            std = filtered_df[numeric_field_id].std()
            normalized_col = f"{numeric_field_id}_normalized"
            filtered_df[normalized_col] = (filtered_df[numeric_field_id] - mean) / std
            print(f"Normalized {numeric_field_id} for filtered records:")
            display(filtered_df[[numeric_field_id, normalized_col]].head())

            # Grouping by a potential group field
            if group_field_id and group_field_id in filtered_df:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
                print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
                display(grouped_df.head())
            break
    else:
        print("Could not find any suitable numeric fields for EDA in the loaded data.")

## 5. Visualization

Visualize the distribution of the selected numeric field (if found). This example uses a histogram.

In [ ]:
# Visualization: Histogram of the numeric field (if available)
import matplotlib.pyplot as plt

if 'filtered_df' in locals() and numeric_field_id is not None and not filtered_df.empty:
    plt.figure(figsize=(8, 5))
    plt.hist(filtered_df[numeric_field_id].dropna(), bins=20, alpha=0.7, color='teal')
    plt.title(f"Distribution of field @id: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
else:
    print("No suitable numeric field found for visualization.")

## 6. Conclusion
In this notebook, we demonstrated:
- How to load a Croissant-described dataset using `mlcroissant`, referencing all entities by their `@id` fields.
- How to enumerate record sets and fields, extract and analyze records, and normalize data.
- Basic exploratory and visualization steps for downstream analysis.

*Note: This dataset currently does not define record sets usable for data extraction. If future versions of the dataset supply record sets, the same code can be applied directly by substituting appropriate `@id` references as shown above.*